In [1]:
# Functions to Compute SL and Take Profit for Short Positions

# Original Stock Shorting Levels
def calculate_original_levels_short(stock_price, atr, sl_mult_1, tp_mult_1, sl_mult_2=None, tp_mult_2=None):
    """
    Calculates Short entry Stop Loss (SL) and Take Profit (TP) price targets.
    For a short position: SL sits above entry, TP sits below entry.
    """
    levels = {
        'SL1': round(stock_price + (sl_mult_1 * atr), 2),
        'TP1': round(stock_price - (tp_mult_1 * atr), 2)
    }
    
    # Check if a second tier of risk/reward levels is requested
    if sl_mult_2 is not None and tp_mult_2 is not None:
        levels['SL2'] = round(stock_price + (sl_mult_2 * atr), 2)
        levels['TP2'] = round(stock_price - (tp_mult_2 * atr), 2)
        
    return levels

# Inverse Stock Long Levels ( when original is short)
def calculate_inverse_levels_short(inverse_entry, stock_entry, original_levels, leverage=1):
    """
    Translates underlying stock targets to an inverse Long ETP position.
    - Underlying increases (hits SL) -> Inverse ETP drops to ETP SL.
    - Underlying decreases (hits TP) -> Inverse ETP rises to ETP TP.
    """
    inverse_levels = {}
    
    for level_name, original_price in original_levels.items():
        if 'SL' in level_name:
            # Underlying stock moved UP: Inverse ETP moves DOWN
            pct_change = (original_price - stock_entry) / stock_entry
            inverse_levels[level_name] = round(inverse_entry * (1 - (leverage * pct_change)), 2)
            
        elif 'TP' in level_name:
            # Underlying stock moved DOWN: Inverse ETP moves UP
            pct_change = (stock_entry - original_price) / stock_entry
            inverse_levels[level_name] = round(inverse_entry * (1 + (leverage * pct_change)), 2)
            
    return inverse_levels



In [2]:
def calculate_long_levels(stock_price, atr, sl_mult_1, tp_mult_1, sl_mult_2=None, tp_mult_2=None):
    """
    Calculates Long entry Stop Loss (SL) and Take Profit (TP) price targets.
    For a long position: SL sits below entry, TP sits above entry.
    """
    levels = {
        'SL1': round(stock_price - (sl_mult_1 * atr), 2),
        'TP1': round(stock_price + (tp_mult_1 * atr), 2)
    }
    
    if sl_mult_2 is not None and tp_mult_2 is not None:
        levels['SL2'] = round(stock_price - (sl_mult_2 * atr), 2)
        levels['TP2'] = round(stock_price + (tp_mult_2 * atr), 2)
        
    return levels


In [3]:
def calculate_short_inverse_levels(inverse_entry, stock_entry, original_long_levels, leverage=1):
    """
    Translates underlying Long targets to a Short Inverse ETP position.
    - Underlying increases (hits TP) -> Inverse ETP drops (ETP TP target achieved).
    - Underlying decreases (hits SL) -> Inverse ETP rises (ETP SL hit).
    """
    inverse_levels = {}
    
    for level_name, original_price in original_long_levels.items():
        if 'SL' in level_name:
            # Underlying stock moved DOWN: Inverse ETP moves UP (This is your ETP Stop Loss)
            pct_change = (stock_entry - original_price) / stock_entry
            inverse_levels[level_name] = round(inverse_entry * (1 + (leverage * pct_change)), 2)
            
        elif 'TP' in level_name:
            # Underlying stock moved UP: Inverse ETP moves DOWN (This is your ETP Take Profit target)
            pct_change = (original_price - stock_entry) / stock_entry
            inverse_levels[level_name] = round(inverse_entry * (1 - (leverage * pct_change)), 2)
            
    return inverse_levels


In [4]:
def calculate_leveraged_long_levels(leveraged_entry, stock_entry, original_long_levels, leverage=3):
    """
    Translates underlying Long targets to a Long Leveraged ETP position (e.g., TQQQ).
    - Underlying increases (hits TP) -> Leveraged ETP moves UP (ETP TP).
    - Underlying decreases (hits SL) -> Leveraged ETP moves DOWN (ETP SL).
    """
    leveraged_levels = {}
    
    for level_name, original_price in original_long_levels.items():
        if 'SL' in level_name:
            # Underlying stock moved DOWN: Leveraged ETP moves DOWN
            pct_change = (stock_entry - original_price) / stock_entry
            leveraged_levels[level_name] = round(leveraged_entry * (1 - (leverage * pct_change)), 2)
            
        elif 'TP' in level_name:
            # Underlying stock moved UP: Leveraged ETP moves UP
            pct_change = (original_price - stock_entry) / stock_entry
            leveraged_levels[level_name] = round(leveraged_entry * (1 + (leverage * pct_change)), 2)
            
    return leveraged_levels


In [5]:
def calculate_option_exit_prices(
    stock_entry: float,
    option_entry: float,
    stock_tp1: float,
    stock_sl1: float,
    delta: float,
    side: str,              # "CALL" or "PUT"
    stock_tp2: float = None,
    stock_sl2: float = None,
    theta: float = 0.0,     # Expected daily theta (can be passed as positive or negative)
    gamma: float = 0.0,     # HIGHLY RECOMMENDED for ATR-sized target moves
    days_held: int = 0
) -> dict:
    """
    Calculates the exact premium price to close an option trade 
    based on underlying stock Take Profit (TP) and Stop Loss (SL) targets.
    Incorporates Delta, Gamma (curvature), and Theta (time decay).
    """
    side = side.upper()
    if side not in ["CALL", "PUT"]:
        raise ValueError("Side must be 'CALL' or 'PUT'")

    # Ensure Delta is handled correctly: Calls are positive, Puts are negative.
    abs_delta = abs(delta)
    effective_delta = abs_delta if side == "CALL" else -abs_delta
    
    # Time decay reduces the contract premium over time.
    total_decay = abs(theta) * days_held

    def get_option_price(stock_target_price: float) -> float:
        stock_move = stock_target_price - stock_entry
        
        # 1. Linear approximation (Delta)
        delta_impact = stock_move * effective_delta
        
        # 2. Curvature approximation (Gamma)
        # Gamma is positive for both long calls and long puts, accelerating profits and decelerating losses.
        gamma_impact = 0.5 * gamma * (stock_move ** 2)
        
        # Total theoretical change in the option's value
        option_price_change = delta_impact + gamma_impact
        
        # Baseline contract premium before time decay
        estimated_premium = option_entry + option_price_change
        
        # Deduct time decay (hurts buyers, helps sellers)
        final_premium = estimated_premium - total_decay
        
        # Options premium cannot drop below $0.01 in the real market
        return round(max(0.01, final_premium), 2)

    # Output dictionary mapping out the limit/stop trigger premiums
    outputs = {
        "option_tp1": get_option_price(stock_tp1),
        "option_sl1": get_option_price(stock_sl1)
    }

    if stock_tp2 is not None and stock_sl2 is not None:
        outputs["option_tp2"] = get_option_price(stock_tp2)
        outputs["option_sl2"] = get_option_price(stock_sl2)  # Fixed the 'outputs_sl2' typo here

    return outputs

In [6]:
crvl = calculate_long_levels(stock_price = 422.50,
                      atr = 18.65,
                      sl_mult_1 = 2.0,
                      tp_mult_1 = 3.0,
                      sl_mult_2=2.5,
                      tp_mult_2=4.0)
print(crvl)

{'SL1': 385.2, 'TP1': 478.45, 'SL2': 375.88, 'TP2': 497.1}


In [7]:
# original_short_levels = calculate_original_levels_short(stock_price,
#                                                         atr,
#                                                         sl_mult_1,
#                                                         tp_mult_1,
#                                                         sl_mult_2=None,
#                                                         tp_mult_2=None)

In [8]:
# # How you call the function:
# exit_prices = calculate_option_exit_prices(
    
#     stock_entry=100.00,
#     option_entry=4.00,
    
#     stock_tp1=95.00,   # Profit target is BELOW
#     stock_sl1=103.00,  # Stop loss is ABOVE
    
#     side="PUT",

#     delta=-0.50,
#     theta = 0.0,     # Expected daily theta (can be passed as positive or negative)
#     gamma = 0.0,     # HIGHLY RECOMMENDED for ATR-sized target moves
#     days_held = 0 # max of 10 for 40 day option and 35 for 60 day option, prefer 45-60 day options
    
# )